# 🔒 3. PII Detection Middleware

LangChain provides built-in `PIIMiddleware` for detecting and handling **Personally Identifiable Information (PII)**.

PII middleware can detect sensitive information in user input and apply different strategies such as redacting, masking, hashing, or blocking it.

---

## 🧩 Supported PII Types

| Type | Example |
|---|---|
| `email` | `user@example.com` |
| `credit_card` | `5105-1051-0510-5100` |
| `ip` | `192.168.1.1` |
| `mac_address` | `00:1A:2B:3C:4D:5E` |
| `url` | `https://secret-site.com` |

---

## 🛠️ PII Handling Strategies

| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |

---

## 🔄 How PII Middleware Works

```text
User Input
    ↓
PII Detection
    ↓
PII Found?
    ├── No  → ✅ Continue
    └── Yes
          ↓
    Apply Strategy
          ↓
    ┌───────────────┐
    │ Redact        │
    │ Mask          │
    │ Hash          │
    │ Block         │
    └───────────────┘

## 🔒 PII Detection Middleware Example

### Import Required Libraries

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

### Create a Dummy Customer Tool

In [4]:
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

### Create Agent with PII Middleware

In [6]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


### Test PII Redaction

In [9]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I’m happy to help! To look up your account securely, could you please confirm the email address you use with us (the full address, not redacted) and the last four digits of the card you mentioned? I’ll use that information only to locate your account and address your request.


In [10]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='a399eba6-55e0-4714-8d85-a160d9f9d2e6'),
  AIMessage(content='I’m happy to help! To look up your account securely, could you please confirm the email address you use with us (the full address, not redacted) and the last\u202ffour digits of the card you mentioned? I’ll use that information only to locate your account and address your request.', additional_kwargs={'reasoning_content': 'The user is providing personal data (email and partial card number). We must not store or process sensitive personal data. We should respond asking for clarification, but we must not request full card number. We can offer help with general info, but we need to verify identity. Possibly we can do a lookup using email? The user gave email (redacted). We can use the function customer_lookup with query maybe email. But we must be careful about pr

### Test API Key Blocking

In [11]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


In [12]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='a399eba6-55e0-4714-8d85-a160d9f9d2e6'),
  AIMessage(content='I’m happy to help! To look up your account securely, could you please confirm the email address you use with us (the full address, not redacted) and the last\u202ffour digits of the card you mentioned? I’ll use that information only to locate your account and address your request.', additional_kwargs={'reasoning_content': 'The user is providing personal data (email and partial card number). We must not store or process sensitive personal data. We should respond asking for clarification, but we must not request full card number. We can offer help with general info, but we need to verify identity. Possibly we can do a lookup using email? The user gave email (redacted). We can use the function customer_lookup with query maybe email. But we must be careful about pr

---

## 💡 Key Idea

> **PIIMiddleware detects sensitive information and applies a predefined strategy before the data reaches the model or agent.**

This helps prevent sensitive information such as emails, credit card numbers, and API keys from being unnecessarily exposed.

---

## 📝 Key Takeaways

- `PIIMiddleware` is a built-in LangChain middleware for handling PII.
- It supports different PII types such as **email, credit cards, IP addresses, MAC addresses, and URLs**.
- Common strategies include **redact, mask, hash, and block**.
- `apply_to_input=True` applies the middleware to incoming user input.
- Custom detectors can be provided using Regex patterns.
- PII middleware is useful for **privacy, security, and compliance**.